In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from ignite.metrics.rec_sys import NDCG

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sn


In [2]:
import gzip
import ast


In [3]:
from pathlib import Path

def load_data(path):
    data = []

    with gzip.open(path, "rt", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                data.append(ast.literal_eval(line))

    return pd.DataFrame(data)

In [ ]:
# Don't download datasets every time
project_dir = Path("..")

processed_dir = project_dir / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

reviews_path = processed_dir / "reviews.parquet"
users_items_path = processed_dir / "users_items.parquet"

if reviews_path.exists() and users_items_path.exists():
    reviews = pd.read_parquet(reviews_path)
    users_items = pd.read_parquet(users_items_path)
    print("Загрузили готовые датафреймы")
else:
    reviews = load_data(project_dir / "data" / "raw" / "australian_user_reviews.json.gz")
    users_items = load_data(project_dir / "data" / "raw" / "australian_users_items.json.gz")

    reviews.to_parquet(reviews_path, index=False)
    users_items.to_parquet(users_items_path, index=False)
    print("Загрузили исходные файлы и сохранили датафреймы")

Загрузили исходные файлы и сохранили датафреймы


In [8]:
print("reviews:", reviews.shape)
display(reviews.head())
display(reviews.info())

print("users_items:", users_items.shape)
display(users_items.head())
display(users_items.info())

reviews: (25799, 3)


,user_id,user_url,reviews
0,76561197970982479,http://steamcommunity.com/profiles/76561197970...,"[{'funny': '', 'posted': 'Posted November 5, 2..."
1,js41637,http://steamcommunity.com/id/js41637,"[{'funny': '', 'posted': 'Posted June 24, 2014..."
2,evcentric,http://steamcommunity.com/id/evcentric,"[{'funny': '', 'posted': 'Posted February 3.',..."
3,doctr,http://steamcommunity.com/id/doctr,"[{'funny': '', 'posted': 'Posted October 14, 2..."
4,maplemage,http://steamcommunity.com/id/maplemage,"[{'funny': '3 people found this review funny',..."


<class 'pandas.DataFrame'>
RangeIndex: 25799 entries, 0 to 25798
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   user_id   25799 non-null  str   
 1   user_url  25799 non-null  str   
 2   reviews   25799 non-null  object
dtypes: object(1), str(2)
memory usage: 2.0+ MB


None

users_items: (88310, 5)


,user_id,items_count,steam_id,user_url,items
0,76561197970982479,277,76561197970982479,http://steamcommunity.com/profiles/76561197970...,"[{'item_id': '10', 'item_name': 'Counter-Strik..."
1,js41637,888,76561198035864385,http://steamcommunity.com/id/js41637,"[{'item_id': '10', 'item_name': 'Counter-Strik..."
2,evcentric,137,76561198007712555,http://steamcommunity.com/id/evcentric,"[{'item_id': '1200', 'item_name': 'Red Orchest..."
3,Riot-Punch,328,76561197963445855,http://steamcommunity.com/id/Riot-Punch,"[{'item_id': '10', 'item_name': 'Counter-Strik..."
4,doctr,541,76561198002099482,http://steamcommunity.com/id/doctr,"[{'item_id': '300', 'item_name': 'Day of Defea..."


<class 'pandas.DataFrame'>
RangeIndex: 88310 entries, 0 to 88309
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   user_id      88310 non-null  str   
 1   items_count  88310 non-null  int64 
 2   steam_id     88310 non-null  str   
 3   user_url     88310 non-null  str   
 4   items        88310 non-null  object
dtypes: int64(1), object(1), str(3)
memory usage: 10.0+ MB


None

In [11]:
print(reviews.iloc[0])
print('-----------------------------------------------------')
print(users_items.iloc[0])

user_id                                     76561197970982479
user_url    http://steamcommunity.com/profiles/76561197970...
reviews     [{'funny': '', 'posted': 'Posted November 5, 2...
Name: 0, dtype: object
-----------------------------------------------------
user_id                                        76561197970982479
items_count                                                  277
steam_id                                       76561197970982479
user_url       http://steamcommunity.com/profiles/76561197970...
items          [{'item_id': '10', 'item_name': 'Counter-Strik...
Name: 0, dtype: object


In [12]:
reviews.loc[0, ["user_id", "reviews"]].to_dict()

{'user_id': '76561197970982479',
 'reviews': [{'funny': '',
   'posted': 'Posted November 5, 2011.',
   'last_edited': '',
   'item_id': '1250',
   'helpful': 'No ratings yet',
   'recommend': True,
   'review': 'Simple yet with great replayability. In my opinion does "zombie" hordes and team work better than left 4 dead plus has a global leveling system. Alot of down to earth "zombie" splattering fun for the whole family. Amazed this sort of FPS is so rare.'},
  {'funny': '',
   'posted': 'Posted July 15, 2011.',
   'last_edited': '',
   'item_id': '22200',
   'helpful': 'No ratings yet',
   'recommend': True,
   'review': "It's unique and worth a playthrough."},
  {'funny': '',
   'posted': 'Posted April 21, 2011.',
   'last_edited': '',
   'item_id': '43110',
   'helpful': 'No ratings yet',
   'recommend': True,
   'review': 'Great atmosphere. The gunplay can be a bit chunky at times but at the end of the day this game is definitely worth it and I hope they do a sequel...so buy the 

In [13]:
reviews_flat = reviews.explode("reviews", ignore_index=True)

review_details = pd.json_normalize(reviews_flat["reviews"])
reviews_flat = pd.concat(
    [reviews_flat[["user_id"]], review_details],
    axis=1
)

display(reviews_flat.head())

,user_id,funny,posted,last_edited,item_id,helpful,recommend,review
0,76561197970982479,,"Posted November 5, 2011.",,1250,No ratings yet,True,Simple yet with great replayability. In my opi...
1,76561197970982479,,"Posted July 15, 2011.",,22200,No ratings yet,True,It's unique and worth a playthrough.
2,76561197970982479,,"Posted April 21, 2011.",,43110,No ratings yet,True,Great atmosphere. The gunplay can be a bit chu...
3,js41637,,"Posted June 24, 2014.",,251610,15 of 20 people (75%) found this review helpful,True,I know what you think when you see this title ...
4,js41637,,"Posted September 8, 2013.",,227300,0 of 1 people (0%) found this review helpful,True,For a simple (it's actually not all that simpl...
